In [ ]:
Modified by JJF on 7/23/2025

#  Machine-learning applications in computational chemistry
## 0. Environment setup 

In [1]:
import psi4
from sklearn.linear_model import LinearRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from oo_cqed_rhf import CQEDRHFCalculator
psi4.core.be_quiet()

At a glance, the energy seems to be a simple function of the bond length $r$. One could formulate this as a simple regression problem and solve for a function that maps $r$ into the energy $E$. However, if one wishes to explore more than the symmetric stretch, more information about the molecule is necessary. We want to describe every bond in the molecule.

To do this, we can encode the geometry of the molecule into a matrix using each atom distance $r_{ij}$ - that is, the distance between every pair of atoms. To differentiate the O-H bond length from the H-H "bond", we will use the atomic charge $Z$ of each atom. The resulting matrix is called a **Coulomb matrix** $\bf C$, which takes the form:
$$
\begin{align}
C_{ij} = 
\begin{cases}
\frac{Z_iZ_j}{r_{ij}}&\text{for $i \neq j$}\\
0.5Z_i^{2.4}&\text{for $i=j$}\\
\end{cases}
\end{align}
$$

If we want to include cavity effects, we need to have some information about the orientation of the electric field associated with
the cavity mode relative to the molecule.  There are a variety of ways we might imagine doing this, but one way would just to be to capture 
the projection of each atom onto the vacuum field of the cavity weighted by the atomic charge.  For a cavity mode characterized by a field vector 

$$
\begin{align}
\mathbf{\lambda} = 
\begin{bmatrix}
\lambda_x \\
\lambda_y \\
\lambda_z
\end{bmatrix}
\end{align}
$$
and an atom with charge $Z$ at position 
$$
\begin{align}
\mathbf{R} = 
\begin{bmatrix}
x \\
y \\
z
\end{bmatrix}
\end{align}
$$

we could incorporate the dot product of the field vector with the atomic coordinate scaled by the atomic charge into the diagonals of the Coulomb matrix as follows:
$$
\begin{align}
C_{ij} = 
\begin{cases}
\frac{Z_iZ_j}{r_{ij}}&\text{for $i \neq j$}\\
0.5Z_i^{2.4} + \alpha \left( Z_i \mathbf{R}_i \cdot \boldsymbol{\lambda} \right)^2 &\text{for $i=j$}\\
\end{cases}
\end{align}
$$

**Note** It is important to carefully consider the magnitude of the cavity terms relative to the existing terms in the Coulomb matrix.  If they are too large, the cavity effects will skew the results too considerably.  If they are too small, the model will likely not account for them in its predictions.  I have included an adjustable scaling factor $\alpha$ that you can initially set to 1, but adjust to be larger or smaller as needed.

**Next step** Modify the `coulomb(geom, q)` function below to include the cavity terms.  You will need to provide it additional arguments for $\lambda$ and $\alpha$, i.e. `coulomb(geom, q, lambda_vector, alpha)`

In [2]:
def coulomb_matrix(geom: np.ndarray,
                   charges: np.ndarray,
                   lambda_vec: np.ndarray,
                   alpha: float = 50.0) -> np.ndarray:
    """
    Build the Coulomb matrix augmented by the cavity term for one geometry.

    Args:
        geom        : (n_atoms, 3) array of Cartesian coords
        charges     : (n_atoms,) array of nuclear charges Z_i
        lambda_vec  : (3,)  cavity polarization vector λ
        alpha       : scaling factor for cavity term

    Returns:
        cm          : (n_atoms, n_atoms) Coulomb matrix
    """
    n = charges.size
    cm = np.zeros((n, n))
    # off‐diagonals
    for i in range(n):
        for j in range(i+1, n):
            Rij = np.linalg.norm(geom[i] - geom[j])
            val = charges[i] * charges[j] / Rij
            cm[i, j] = cm[j, i] = val

    # diagonals
    for i in range(n):
        Zi = charges[i]
        dot = geom[i].dot(lambda_vec)
        cm[i, i] = 0.5 * Zi**2.4 + alpha * (Zi * dot)**2

    return cm

def build_dataset(mol_geoms: list[np.ndarray],
                  mol_charges: list[np.ndarray],
                  lambdas:   list[np.ndarray],
                  energies:  list[float],
                  flatten:   bool = True) -> tuple[np.ndarray, np.ndarray]:
    """
    Turn lists of (geom, charges, λ) and energies into X, y arrays.
    """
    X = []
    for geom, q, lam in zip(mol_geoms, mol_charges, lambdas):
        cm = coulomb_matrix(geom, q, lam)
        X.append(cm.flatten() if flatten else cm)
    return np.vstack(X), np.array(energies)

def split_dataset(X: np.ndarray,
                  y: np.ndarray,
                  train_size: float = 0.2,
                  random_state: int = 42):
    """
    Convenience wrapper around sklearn.train_test_split.
    """
    return train_test_split(
        X, y,
        train_size=train_size,
        random_state=random_state
    )

def generate_h2o_stretch(rs: np.ndarray,
                         lambda_vectors: list[np.ndarray] | np.ndarray,
                         psi4_opts: dict
                         ) -> tuple[list[np.ndarray],
                                    list[np.ndarray],
                                    list[np.ndarray],
                                    list[float]]:
    """
    Generate (geom, charge, lambda, energy) for every combination
    of bond length in `rs` and cavity vector in `lambda_vectors`.
    """
    geoms, qs, ls, Es = [], [], [], []

    # ensure it's a simple Python list of 3-vectors
    lam_list = list(lambda_vectors)

    for r1 in rs:
        for r2 in rs:
            # build the shared Z-matrix string for this bond length
            mol_str = f"""
            O
            H 1 {r1}
            H 1 {r2} 2 104.5
            symmetry c1
            """
    
            for lam in lam_list:
                print(F"lambda vector is \n {lam}")
                # --- build molecule and record geometry & charges ---
                mol = psi4.geometry(mol_str)
                geoms.append(mol.geometry().to_array())
                qs.append(np.array([mol.fZ(i) for i in range(mol.natom())]))
                ls.append(lam.copy())
    
                # --- compute and record CQED-RHF energy for this λ ---
                calc = CQEDRHFCalculator(lam, mol_str, psi4_opts)
                calc.calc_cqed_rhf_energy()
                print(F" r1 is {r1} r2 is {r2} lz is {lam[2]} and E is {calc.cqed_rhf_energy}")
                Es.append(calc.cqed_rhf_energy)

    return geoms, qs, ls, Es


In [3]:
# parameters
rs = np.linspace(0.5, 1.2, 10)
lambda_vecs = [np.array([0.0, 0.0, lz])  for lz in np.linspace(0.0, 0.1, 10)]
print(rs)
print(lambda_vecs)
# (you can mix-and-match: e.g. Cartesian grid of angles & magnitudes)

psi4_options = {
    "basis": "sto-3g",
    "save_jk": True,
    "scf_type": "pk",
    "e_convergence": 1e-12,
    "d_convergence": 1e-12,
}
psi4.set_options(psi4_options)



[0.5        0.57777778 0.65555556 0.73333333 0.81111111 0.88888889
 0.96666667 1.04444444 1.12222222 1.2       ]
[array([0., 0., 0.]), array([0.        , 0.        , 0.01111111]), array([0.        , 0.        , 0.02222222]), array([0.        , 0.        , 0.03333333]), array([0.        , 0.        , 0.04444444]), array([0.        , 0.        , 0.05555556]), array([0.        , 0.        , 0.06666667]), array([0.        , 0.        , 0.07777778]), array([0.        , 0.        , 0.08888889]), array([0. , 0. , 0.1])]


In [4]:
geoms, qs, lambdas, Es = generate_h2o_stretch(rs, lambda_vecs, psi4_options)

lambda vector is 
 [0. 0. 0.]
 r1 is 0.5 r2 is 0.5 lz is 0.0 and E is -73.12423439178012
lambda vector is 
 [0.         0.         0.01111111]
 r1 is 0.5 r2 is 0.5 lz is 0.011111111111111112 and E is -73.12407699687631
lambda vector is 
 [0.         0.         0.02222222]
 r1 is 0.5 r2 is 0.5 lz is 0.022222222222222223 and E is -73.12360481549844
lambda vector is 
 [0.         0.         0.03333333]
 r1 is 0.5 r2 is 0.5 lz is 0.03333333333333333 and E is -73.12281785764642
lambda vector is 
 [0.         0.         0.04444444]
 r1 is 0.5 r2 is 0.5 lz is 0.044444444444444446 and E is -73.12171613998329
lambda vector is 
 [0.         0.         0.05555556]
 r1 is 0.5 r2 is 0.5 lz is 0.05555555555555556 and E is -73.12029968582986
lambda vector is 
 [0.         0.         0.06666667]
 r1 is 0.5 r2 is 0.5 lz is 0.06666666666666667 and E is -73.11856852515794
lambda vector is 
 [0.         0.         0.07777778]
 r1 is 0.5 r2 is 0.5 lz is 0.07777777777777778 and E is -73.11652269458045
lambd

In [9]:

n_r = len(rs)
n_l = len(lambda_vecs)

# ----- 2) Build and split dataset -----
X, y = build_dataset(geoms, qs, lambdas, Es)

# 2) create flat index array 0…N-1
flat_idx = np.arange(n_r * n_r * n_l)

# 3) split everything together
X_train, X_test, \
y_train, y_test, \
idx_train, idx_test = train_test_split(
    X, y, flat_idx,
    train_size=0.30,
    random_state=1,
    shuffle=True
)

# 4) unravel the flat indices back into (r1-index, r2-index, λ-index)
r1_train_idx, r2_train_idx, l_train_idx = np.unravel_index(idx_train, (n_r, n_r, n_l))
r1_test_idx,  r2_test_idx, l_test_idx  = np.unravel_index(idx_test,  (n_r, n_r, n_l))

# now you can ask:
train_r1   = rs[r1_train_idx]
train_r2   = rs[r2_train_idx]
train_lams = [lambda_vecs[i] for i in l_train_idx]

test_r1    = rs[r1_test_idx]
test_r2    = rs[r2_test_idx]
test_lams  = [lambda_vecs[i] for i in l_test_idx]

print("Train set has", len(train_r1), "points:")
for r1_val, r2_val, lam in zip(train_r1, train_r2, train_lams):
    print(f"  r={r1_val:.3f} Å, r={r2_val:.3f} Å, λ={lam}")

print("\nTest set has", len(test_r1), "points:")
for r1_val, r2_val, lam in zip(test_r1, test_r2, test_lams):
    print(f"  r={r1_val:.3f} Å, r={r2_val:.3f} Å, λ={lam}")



Train set has 300 points:
  r=0.733 Å, r=0.656 Å, λ=[0.         0.         0.06666667]
  r=0.811 Å, r=0.889 Å, λ=[0.         0.         0.08888889]
  r=0.967 Å, r=1.122 Å, λ=[0.         0.         0.01111111]
  r=0.967 Å, r=0.967 Å, λ=[0.         0.         0.02222222]
  r=1.044 Å, r=0.967 Å, λ=[0.         0.         0.02222222]
  r=0.811 Å, r=0.967 Å, λ=[0.         0.         0.05555556]
  r=0.578 Å, r=0.967 Å, λ=[0.         0.         0.03333333]
  r=0.578 Å, r=0.967 Å, λ=[0.         0.         0.07777778]
  r=0.578 Å, r=0.811 Å, λ=[0.         0.         0.05555556]
  r=0.656 Å, r=0.500 Å, λ=[0.         0.         0.06666667]
  r=0.889 Å, r=0.500 Å, λ=[0. 0. 0.]
  r=0.578 Å, r=1.122 Å, λ=[0.         0.         0.08888889]
  r=0.733 Å, r=0.656 Å, λ=[0.         0.         0.04444444]
  r=0.967 Å, r=0.889 Å, λ=[0.         0.         0.07777778]
  r=1.044 Å, r=0.811 Å, λ=[0. 0. 0.]
  r=1.122 Å, r=0.733 Å, λ=[0. 0. 0.]
  r=1.122 Å, r=0.889 Å, λ=[0. 0. 0.]
  r=1.200 Å, r=1.200 Å, λ=[0.    

In [13]:
# ----- 3) Train KRR with grid-search CV -----
param_grid = {
    'alpha': np.logspace(-12, 12, num=12),
    'gamma': np.logspace(-12, 12, num=12)
}

krr = GridSearchCV(KernelRidge(kernel='rbf'),
                   param_grid,
                   cv=10,
                   scoring='neg_mean_squared_error')
krr.fit(X_train, y_train)
y_pred = krr.predict(X_test)

from sklearn.metrics import mean_absolute_error

# compute MAE
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean absolute error on test set: {mae * 627:.6f}  kcal / mol")


Mean absolute error on test set: 2.416249  kcal / mol


In [15]:

# 1) Define a grid over alpha, gamma, and coef0 (offset in tanh)
param_grid = {
    'alpha':  np.logspace(-12, 12, num=12),
    'gamma':  np.logspace(-12, 12, num=12),
    'coef0':  np.linspace(0, 1, num=5)    # try offsets between 0 and 1
}

# 2) Create your GridSearchCV with the sigmoid kernel
krr_sigmoid = GridSearchCV(
    KernelRidge(kernel='sigmoid'),
    param_grid,
    cv=10,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

# 3) Fit & predict as before
krr_sigmoid.fit(X_train, y_train)
y_pred = krr_sigmoid.predict(X_test)

# 4) Inspect best‐found hyperparameters
print("Best params:", krr_sigmoid.best_params_)

/Users/jfoley19/miniconda3/envs/psi4_new/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/Users/jfoley19/miniconda3/envs/psi4_new/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/Users/jfoley19/miniconda3/envs/psi4_new/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/Users/jfoley19/miniconda3/envs/psi4_new/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/Users/jfoley19/miniconda3/envs/psi4_new/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:252: UserWarning: Singular matrix in so

Best params: {'alpha': 1.5199110829529332e-10, 'coef0': 0.25, 'gamma': 0.0005336699231206301}


/Users/jfoley19/miniconda3/envs/psi4_new/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(


In [16]:
# compute MAE
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean absolute error on test set: {mae * 627:.6f}  kcal / mol")


Mean absolute error on test set: 2.445227  kcal / mol


In [14]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error
import numpy as np

# --- 1) Define your hyperparameter grid for the MLP ---
param_grid = {
    # try one- and two-layer nets of various widths
    'hidden_layer_sizes': [(50,), (100,), (100, 50), (100, 100)],  
    # activations to consider
    'activation': ['relu', 'tanh'],                            
    # L2 penalty
    'alpha': [1e-5, 1e-4, 1e-3],                                
    # initial learning rate
    'learning_rate_init': [1e-4, 1e-3],                         
}

# --- 2) Set up the GridSearchCV on an MLPRegressor ---
mlp = MLPRegressor(max_iter=5000,            # allow enough epochs
                   random_state=1,           # reproducibility
                   early_stopping=True,      # stop if no improvement
                   n_iter_no_change=50)      # patience for early stop

mlp_search = GridSearchCV(
    mlp,
    param_grid,
    cv=5,                                     # 5-fold CV
    scoring='neg_mean_squared_error',
    n_jobs=-1                                # parallelize if you can
)

# --- 3) Fit on your training set ---
mlp_search.fit(X_train, y_train)

# best params
print("Best MLP params:", mlp_search.best_params_)

# --- 4) Predict on the test set ---
y_pred = mlp_search.predict(X_test)

# --- 5) Compute MAE in Hartree and convert to kcal/mol (1 Eh ≈ 627.509 kcal/mol) ---
mae_hartree = mean_absolute_error(y_test, y_pred)
mae_kcal_mol = mae_hartree * 627.509
print(f"Test MAE: {mae_hartree:.6f} Eh  ≃  {mae_kcal_mol:.4f} kcal/mol")


/Users/jfoley19/miniconda3/envs/psi4_new/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/jfoley19/miniconda3/envs/psi4_new/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/jfoley19/miniconda3/envs/psi4_new/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/jfoley19/miniconda3/envs/psi4_new/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5000) reached and the optimization

Best MLP params: {'activation': 'tanh', 'alpha': 1e-05, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001}
Test MAE: 0.367273 Eh  ≃  230.4669 kcal/mol
